In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from numpy import random as rand
from numpy import linalg as lin
import random
from cs771 import plotData as pd
from cs771 import utils
import struct
from array import array
from os.path import join
import matplotlib.pyplot as plt

In [3]:
#
# MNIST Data Loader Class
#
class MnistDataloader(object):
    def __init__(self, training_images_filepath,training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath
    
    def read_images_labels(self, images_filepath, labels_filepath):        
        labels = []
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError('Magic number mismatch, expected 2049, got {}'.format(magic))
            labels = array("B", file.read())        
        
        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError('Magic number mismatch, expected 2051, got {}'.format(magic))
            image_data = array("B", file.read())        
        images = []
        for i in range(size):
            images.append([0] * rows * cols)
        for i in range(size):
            img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
            img = img.reshape(28, 28)
            images[i][:] = img            
        
        return images, labels
            
    def load_data(self):
        x_train, y_train = self.read_images_labels(self.training_images_filepath, self.training_labels_filepath)
        x_test, y_test = self.read_images_labels(self.test_images_filepath, self.test_labels_filepath)
        return (x_train, y_train),(x_test, y_test) 

In [4]:
#
# Set file paths based on added MNIST Datasets
#
input_path = 'mnist/'
training_images_filepath = join(input_path, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
training_labels_filepath = join(input_path, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
test_images_filepath = join(input_path, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
test_labels_filepath = join(input_path, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

In [5]:
mnist_dataloader = MnistDataloader(training_images_filepath, training_labels_filepath, test_images_filepath, test_labels_filepath)
((XTrain, yTrain), (XTest, yTest)) = mnist_dataloader.load_data()
yTrain = np.array( yTrain ) 
yTest = np.array( yTest )

tmp = np.zeros(( len( XTrain ), 28, 28 ))
for i in range( len( XTrain )):
    tmp[ i, : ] = np.concatenate( XTrain[i] ).reshape( ( 28, 28 ) )
XTrain = tmp


In [6]:
from tqdm import tqdm


def run_kmeans(data, K, max_iters=100, random_seed=42):
    """
    Implements a simple K-Means clustering algorithm using only NumPy.

    Args:
        data (np.ndarray): N x D array of data points.
        K (int): The number of clusters.
        max_iters (int): Maximum number of iterations for convergence.
        random_seed (int): Seed for reproducibility.

    Returns:
        tuple: (centroids, labels)
            centroids (np.ndarray): K x D array of final cluster centers.
            labels (np.ndarray): N-dimensional array of cluster assignments (0 to K-1).
    """
    N, D = data.shape
    np.random.seed(random_seed)

    # 1. Initialize centroids (randomly select K data points)
    initial_indices = np.random.choice(N, K, replace=False)
    centroids = data[initial_indices]

    labels = np.zeros(N, dtype=int)

    for _ in tqdm(range(max_iters), desc="K-Means Clustering"):
        # Store old labels to check for convergence
        labels_old = np.copy(labels)
        
        # E-Step (Assignment): Assign each data point to the closest centroid
        # Calculate squared Euclidean distance between all points and all centroids
        # N x K matrix of squared distances
        
        # Expand dimensions for broadcasting (N x 1 x D) - (1 x K x D)
        # Resulting distance matrix is N x K
        distances = np.sum((data[:, np.newaxis, :] - centroids[np.newaxis, :, :])**2, axis=2)
        
        # Assign label based on minimum distance
        labels = np.argmin(distances, axis=1)

        # M-Step (Update): Recalculate centroids
        centroids_new = np.zeros((K, D))
        
        for k in range(K):
            # Find all data points belonging to the k-th cluster
            cluster_data = data[labels == k]
            
            if len(cluster_data) > 0:
                # Calculate the new mean of the cluster
                centroids_new[k] = np.mean(cluster_data, axis=0)
            else:
                # If a cluster is empty, keep the old centroid to prevent crashes
                centroids_new[k] = centroids[k]
        
        # Update centroids
        centroids = centroids_new
        
        # Check for convergence (labels don't change)
        if np.array_equal(labels, labels_old):
            # print(f"K-Means converged after {iteration + 1} iterations.")
            break

    return centroids, labels


In [7]:
def initialize_gmm_params(data_c, K, reg_covar=1e-6):
    """
    Initializes the GMM parameters (means, covariances, mixing coefficients) 
    for a single class 'c' using the K-Means algorithm.

    The assignment specifies that only libraries imported in the lecture notebook 
    should be used. Since a full KMeans implementation is complex, this function 
    uses sklearn.cluster.KMeans for robustness, assuming it is either permitted 
    or can be replaced by a simple custom K-Means function if necessary.

    Args:
        data_c (np.ndarray): N_c x D array of data points belonging to class c.
        K (int): The number of Gaussian components (clusters).
        reg_covar (float): Regularization term added to the diagonal of the 
                           covariance matrices to ensure they are invertible.

    Returns:
        tuple: (pi_c, mu_c, Sigma_c)
            pi_c (np.ndarray): K-dimensional vector of mixing coefficients (pi^c).
            mu_c (np.ndarray): K x D matrix of component means (mu^c_k).
            Sigma_c (np.ndarray): K x D x D array of component covariance matrices (Sigma^c_k).
    """
    
    N_c, D = data_c.shape
    
    # 1. Run K-Means Clustering on the class data
    centroids, labels = run_kmeans(data_c, K, max_iters=100)
    

    # Initialize storage for GMM parameters
    pi_c = np.zeros(K)
    mu_c = centroids  # K x D
    Sigma_c = np.zeros((K, D, D))  # K x D x D

    # 2. Calculate initial pi and Sigma based on K-Means results

    for k in range(K):
        # Find all data points belonging to the k-th cluster
        cluster_data = data_c[labels == k]
        N_k = len(cluster_data)

        # Handle empty clusters (can occur in K-Means)
        if N_k == 0:
            print(f"Warning: Cluster {k} is empty. Using fallback initialization.")
            # If empty, assign small mixing weight and use global mean/cov
            pi_c[k] = 1e-6 # A tiny value
            # Fall back to overall mean for this component
            mu_c[k] = np.mean(data_c, axis=0) 
            # Use global covariance
            Sigma_c[k] = np.cov(data_c, rowvar=False)
            
        else:
            # A. Calculate initial mixing coefficient (pi^c_k)
            pi_c[k] = N_k / N_c
            
            # B. Calculate initial component covariance (Sigma^c_k)
            # Use the cluster data to estimate covariance
            if N_k > 1:
                cov_k = np.cov(cluster_data, rowvar=False)
            else:
                # If only one point, fall back to global covariance
                cov_k = np.cov(data_c, rowvar=False)
                
            # Add regularization for numerical stability (required for invertible matrix)
            Sigma_c[k] = cov_k + reg_covar * np.eye(D)

    # 3. Normalize pi_c to ensure sum-to-one (important if empty clusters occurred)
    pi_c = pi_c / np.sum(pi_c)

    return pi_c, mu_c, Sigma_c

In [8]:
# Learn a single (general) Gaussian per class
def learnClassConditionalDist( X, y ):
    (labels, labelCounts) = np.unique( y, return_counts = True )
    C = labels.size
    (n, d) = X.shape
    muVals = np.zeros( (C, d) )
    SigmaVals = np.zeros( (C, d, d) )
    for c in range( C ):
        XThisLabel = X[y == c]
        # MLE for mean is simply the empirical mean (aka sample mean) of feature vectors of this class
        muVals[c] = np.mean( XThisLabel, axis = 0 )
        XCent = XThisLabel - muVals[c]
        # MLE for covariance matrix is simply the empirical/sample covariance matrix of this class
        SigmaVals[c] = 1/labelCounts[c]*(XCent.T).dot( XCent )
    return (C, list( zip( muVals, SigmaVals, np.arange( C ), labelCounts/n ) ))

# Stretch out the 28 x 28 image into a 784 dimensional vector
def flattenTensor( X ):
    n = X.shape[0]
    d = np.prod( X.shape[1:] )
    return X.reshape( n, d )

def predictClassScores( X, mu, Sigma, p, mask = [] ):
    # If no mask was provided, then we need to consider all coordinates of the feature vector
    if len(mask) == 0:
        mask = np.ones( (X.shape[1],), dtype=bool )
    
    XCent = X[:,mask] - mu[mask]
    # The covariance matrix will mostly be non-invertible in this case
    # This is because some pixels are always white in every image i.e. zero variance
    # The Moore-Penrose "pseudoinverse" is used in such situations
    SInv = lin.pinv( Sigma[mask,:][:,mask] )
    # numpy has a helpful routine that directly calculates the logdet of a matrix
    # More numerically stable in several situations -- no exploding outputs
    (sign, logdet) = lin.slogdet( Sigma[mask,:][:,mask] )
    # sign = 0 is the routines way of telling us that the determinant was zero
    # The determinant of a covariance matrix should always be non-negative
    if sign <= 0:
        SLogDet = 0
    else:
        SLogDet = logdet
    # This term gives us (ignoring additive constants) ln P[x | y, \theta]
    term1 = 0.5 * (-SLogDet - np.sum( np.multiply( np.dot(XCent, SInv), XCent), axis = 1 ))
    # This term gives us ln P[y | \theta]
    term2 = np.log( p )
    return term1 + term2

def predictGen( X, model, C, mask = [] ):
    classScores = np.zeros( (X.shape[0], C) )
    for mu, Sigma, c, p in model:
        classScores[:,c] = predictClassScores( X, mu, Sigma, p, mask )
    return np.argmax( classScores, axis = 1 )

def gaussianPDF(X, mu, Sigma):
    """
    Computes the *logarithm* of the probability density function (PDF)
    for a multivariate Gaussian. This is numerically stable.

    log(N(x | mu, Sigma)) = -D/2*log(2pi) - 1/2*log|Sigma| - 1/2*(x-mu)^T*Sigma^-1*(x-mu)

    Args:
        X (np.ndarray): N x D data matrix.
        mu (np.ndarray): D-dimensional mean vector.
        Sigma (np.ndarray): D x D covariance matrix.

    Returns:
        np.ndarray: N-dimensional vector of log-PDF values for each data point.
    """
    N, D = X.shape
    
    # Calculate the log-determinant and inverse of the covariance matrix
    try:
        # np.linalg.slogdet returns (sign, log(abs(det)))
        # We need log(det), so we take the second element.
        # Sign must be positive for a valid covariance matrix.
        sign, log_det_Sigma = np.linalg.slogdet(Sigma)
        if sign != 1:
             # This can happen with numerical instability
             # Return a very small log-probability
             return np.full(N, -1e100) # Extremely small log-prob

        Sigma_inv = np.linalg.inv(Sigma)
    except np.linalg.LinAlgError:
        # Fallback for singular matrix
        print("Warning: Singular matrix in log_gaussian_pdf")
        return np.full(N, -1e100)

    # Calculate the log of the normalization constant
    log_norm_const = -0.5 * D * np.log(2 * np.pi) - 0.5 * log_det_Sigma

    # Calculate the exponent term: (x - mu)^T * Sigma^-1 * (x - mu)
    X_minus_mu = X - mu  # N x D

    # 1. Compute (x - mu) * Sigma^-1
    # Result is N x D
    term1 = np.dot(X_minus_mu, Sigma_inv)

    # 2. Compute ( (x - mu) * Sigma^-1 ) * (x - mu)^T 
    # Use np.sum(..., axis=1) for the equivalent of taking the diagonal of the matrix product
    # Result is N-dimensional vector of quadratic forms
    quadratic_form = np.sum(term1 * X_minus_mu, axis=1) 

    # Combine terms
    log_pdf_values = log_norm_const - 0.5 * quadratic_form
    
    return log_pdf_values

((XTrain, yTrain), (XTest, yTest)) = mnist_dataloader.load_data()
yTrain = np.array( yTrain )
yTest = np.array( yTest )

tmp = np.zeros( ( len( XTrain ), 28, 28 ) )
for i in range( len( XTrain )):
    tmp[ i, : ] = np.concatenate( XTrain[i] ).reshape( ( 28, 28 ) )
XTrain = tmp

tmp = np.zeros( ( len( XTest ), 28, 28 ) )
for i in range( len( XTest )):
    tmp[ i, : ] = np.concatenate( XTest[i] ).reshape( ( 28, 28 ) )
XTest = tmp

In [17]:
# Normalize data coordinates otherwise numbers in later calculations explode
# Also, flatten images for sake of convenience
imShape = XTrain.shape[1:]
XTrainFlat = flattenTensor( XTrain/256 )
XTestFlat = flattenTensor( XTest/256 )

# Number of gaussians per class
K = 5
numClasses = 10

In [ ]:
muValues = np.zeros( (numClasses, K, XTrainFlat.shape[1]) )
sigmaValues = np.zeros((numClasses, K, XTrainFlat.shape[1], XTrainFlat.shape[1]))
piValues = np.zeros( (numClasses, K) )

# Initialize the parameters for each class
for c in range( numClasses ):
    XThisClass = XTrainFlat[ yTrain == c ]
    pi_c, mu_c, Sigma_c = initialize_gmm_params( XThisClass, K )
    muValues[c] = mu_c
    sigmaValues[c] = Sigma_c
    piValues[c] = pi_c


K-Means Clustering:  41%|████      | 41/100 [00:06<00:09,  6.19it/s]


In [10]:
# Store the mu, Sigma, and pi values using pickle
import pickle
with open('gmm_params.pkl', 'wb') as f:
    pickle.dump((muValues, sigmaValues, piValues), f)

In [18]:
import pickle
# Load the parameters from the pickle file
with open('gmm_params.pkl', 'rb') as f:
    muValues, sigmaValues, piValues = pickle.load(f)

In [19]:
def gmm_inference(X_test, mu, sigma, pi, yTrain):
    numClasses = 10
    yPred = np.zeros( ( X_test.shape[0], ), dtype=int )
    classScores = np.zeros( (X_test.shape[0], numClasses) )
    for c in range(numClasses):
        tempScores = np.zeros( (X_test.shape[0], K) )
        num_data_in_class = np.sum(yTrain == c)
        total_data = yTrain.shape[0]
        prior_class = num_data_in_class / total_data
        for k in range(K):
            log_pdf_value = gaussianPDF(X_test, mu[c, k], sigma[c, k])
            tempScores[:, k] = np.log(pi[c, k]) + log_pdf_value
        # Use log-sum-exp trick to combine scores from all components
        max_temp_score = np.max(tempScores, axis=1, keepdims=True)
        classScores[:, c] =  prior_class * np.sum(np.exp(tempScores - max_temp_score), axis=1)
    yPred = np.argmax( classScores, axis=1 )
    return yPred

In [20]:
# Iteration loop for EM Algorithm
max_iteration = 100
inference_interval = 10
for iteration in tqdm(range(max_iteration), desc="GMM EM Algorithm"):

    # Do infenrencing to check model performance
    if iteration % inference_interval == 0:
        # num_samples_to_check = 1000
        # total_samples = yTest.shape[0]
        # rand_index_of_test_samples = random.sample(range(total_samples), num_samples_to_check)
        yPred = gmm_inference(XTestFlat, muValues, sigmaValues, piValues, yTrain)
        accuracy = np.sum(yPred == yTest) / yTest.size
        print(f"Iteration {iteration + 1}, Prediction Accuracy: {accuracy:.4f}")

    # E-Step
    # Calculation of gamma responsibilities for each class
    gamma = np.zeros((XTrainFlat.shape[0], K))  # N x K
    for c in range(numClasses):
        XThisClass = XTrainFlat[ yTrain == c ]
        XThisClass_index = np.where(yTrain == c)[0]  # Get indices of this class
        for k in range(K):
            log_pdf_values = gaussianPDF(XThisClass, muValues[c, k], sigmaValues[c, k])
            gamma[XThisClass_index, k] = np.log(piValues[c, k]) + log_pdf_values
    # 2. Apply Log-Sum-Exp Trick for stabilization
    # Find the maximum log-value for each data point i across all components j
    # This is L_max_i in the derivation
    log_max_per_data_point = np.max(gamma, axis=1, keepdims=True)

    # 3. Subtract the max log-value from each log-weighted PDF for numerical stability
    log_weighted_pdfs = gamma - log_max_per_data_point

    # 4. Compute the new responsibilities
    # Apply the softmax function to get the responsibilities
    gamma = np.exp(log_weighted_pdfs)
    sum_gamma = np.sum(gamma, axis=1, keepdims=True)
    gamma /= sum_gamma

    # M-Step
    for c in range(numClasses):
        XThisClass = XTrainFlat[ yTrain == c ]
        XThisClass_index = np.where(yTrain == c)[0]  # Get indices of this class
        N_c = XThisClass.shape[0]
        
        # Update mixing coefficients
        for k in range(K):
            N_c_k = np.sum(gamma[XThisClass_index, k])
            piValues[c, k] = N_c_k / N_c

        # Update means
        for k in range(K):
            muValues[c, k] = np.sum(gamma[XThisClass_index, k][:, np.newaxis] * XThisClass, axis=0) / np.sum(gamma[XThisClass_index, k])
        
        # Update covariances
        for k in range(K):
            diff = XThisClass - muValues[c, k]
            sigmaValues[c, k] = (gamma[XThisClass_index, k][:, np.newaxis] * diff).T @ diff / np.sum(gamma[XThisClass_index, k])
            sigmaValues[c, k] += 1e-6 * np.eye(XTrainFlat.shape[1])  # Regularization term

# (C, model) = learnClassConditionalDist( XTrainFlat, yTrain )
# yPred = predictGen( XTestFlat, model, C )

# print( "Prediction Accuracy: ", sum(yPred == yTest)/yTest.size )

# numRows = 20
# numCols = 6

# fig13, axs13 = pd.getFigList( numRows, numCols, sizey = 3.2 )
# labels = ["True Label: %s\nPred Label: %s" % (yTest[i], yPred[i]) for i in range( numRows * numCols )]
# pd.showImagesNoAxes( axs13, XTestFlat[:numRows*numCols], numRows, numCols, resize = True, imShape = imShape, labelList = labels )

GMM EM Algorithm:   0%|          | 0/100 [00:00<?, ?it/s]

Iteration 1, Prediction Accuracy: 0.1161


GMM EM Algorithm:  10%|█         | 10/100 [24:12<3:29:38, 139.76s/it]

Iteration 11, Prediction Accuracy: 0.1150


GMM EM Algorithm:  20%|██        | 20/100 [40:33<1:12:47, 54.60s/it] 

...

Iteration 31, Prediction Accuracy: 0.1148


GMM EM Algorithm:  40%|████      | 40/100 [1:26:09<2:18:03, 138.05s/it]

Iteration 41, Prediction Accuracy: 0.1142


GMM EM Algorithm:  40%|████      | 40/100 [1:28:47<2:13:11, 133.20s/it]


KeyboardInterrupt: 

In [12]:
# Store the mu, Sigma, and pi values using pickle
import pickle
with open('gmm_params_final.pkl', 'wb') as f:
    pickle.dump((muValues, sigmaValues, piValues), f)

In [ ]:
# Inference code
# Load the parameters from the pickle file
with open('gmm_params_final.pkl', 'rb') as f:
    muValues, sigmaValues, piValues = pickle.load(f)

((XTrain, yTrain), (XTest, yTest)) = mnist_dataloader.load_data()
yTrain = np.array( yTrain )
yTest = np.array( yTest )
tmp = np.zeros( ( len( XTrain ), 28, 28 ) )
for i in range( len( XTrain )):
    tmp[ i, : ] = np.concatenate( XTrain[i] ).reshape( ( 28, 28 ) )
XTrain = tmp

tmp = np.zeros( ( len( XTest ), 28, 28 ) )
for i in range( len( XTest )):
    tmp[ i, : ] = np.concatenate( XTest[i] ).reshape( ( 28, 28 ) )
XTest = tmp

# Normalize data coordinates otherwise numbers in later calculations explode
imShape = XTrain.shape[1:]
XTrainFlat = flattenTensor( XTrain/256 )
XTestFlat = flattenTensor( XTest/256 )

In [24]:
# i want to test this line
import numpy as np

# XTrainFlat = flattenTensor( XTrain/256 )

# number of data points, number of classes, number of gaussians per class
num_data_points = 5
num_gaussians = 2

gamma = np.random.randint(0, 10, (num_data_points, num_gaussians)).astype(float)

print("Gamma before normalization:")
print(gamma)

print(gamma[:, 0])

# max_values = np.max(gamma, axis=2, keepdims=True)
# log_exp = np.exp(gamma - max_values)
# sum_exp = np.sum(log_exp, axis=2, keepdims=True)
# gamma = log_exp / sum_exp

# print("Gamma after Log-Sum-Exp normalization:")
# print(gamma)

# for i in range(num_classes):
#     for j in range(num_data_points):
#         gamma[j, i, :] /= np.sum(gamma[j, i, :])
# print("Gamma after normalization:")
# print(gamma)


Gamma before normalization:
[[1. 3.]
 [6. 4.]
 [5. 1.]
 [2. 6.]
 [1. 5.]]
[1. 6. 5. 2. 1.]
